In [24]:
#zone A data set
import pandas as pd
import random

data=[]

for i in range(600):

    scenario = random.choices(
        ["safe","border","risk"],
        weights=[6,2,2]   # 60% safe, 20% borderline, 20% risk
    )[0]

    if scenario=="safe":

        avg_v = random.randint(225,240)
        min_v = avg_v - random.randint(0,5)
        avg_c = round(random.uniform(3,4.5),2)
        max_c = avg_c + round(random.uniform(0,0.8),2)
        temp = random.randint(30,36)

        risk = 0

    elif scenario=="border":

        avg_v = random.randint(210,225)
        min_v = avg_v - random.randint(6,12)
        avg_c = round(random.uniform(4.8,6.2),2)
        max_c = avg_c + round(random.uniform(1,1.5),2)
        temp = random.randint(36,42)

        # Sometimes risky, sometimes safe
        risk = random.choice([0,1])

    else:  # risk

        avg_v = random.randint(195,210)
        min_v = avg_v - random.randint(12,18)
        avg_c = round(random.uniform(6.2,7.5),2)
        max_c = avg_c + round(random.uniform(1.5,2.2),2)
        temp = random.randint(42,48)

        risk = 1


    fluct = abs(avg_v-min_v)

    data.append([
        avg_v,min_v,avg_c,max_c,temp,fluct,risk
    ])


df = pd.DataFrame(data,columns=[
    "avg_voltage","min_voltage","avg_current",
    "max_current","temp","fluctuation","risk"
])

df.to_csv("zoneA_data.csv",index=False)

print("Improved Zone A dataset created")


Improved Zone A dataset created


In [64]:
#zone B
import pandas as pd
import random

data = []

for i in range(600):

    scenario = random.choices(
        ["high","very_high","extreme"],
        weights=[4,4,2]   # mostly high & very high
    )[0]


    if scenario == "high":
        # ~70% risk zone
        avg_v = random.randint(200,210)
        min_v = avg_v - random.randint(8,14)
        avg_c = round(random.uniform(6.2,7.0),2)
        max_c = avg_c + round(random.uniform(1.2,1.8),2)
        temp = random.randint(40,45)
        risk = 1


    elif scenario == "very_high":
        # ~80–85% risk zone
        avg_v = random.randint(190,200)
        min_v = avg_v - random.randint(14,20)
        avg_c = round(random.uniform(7.0,8.0),2)
        max_c = avg_c + round(random.uniform(1.8,2.3),2)
        temp = random.randint(45,50)
        risk = 1


    else:  # extreme
        # ~90–95% risk zone
        avg_v = random.randint(175,190)
        min_v = avg_v - random.randint(20,28)
        avg_c = round(random.uniform(8.0,9.2),2)
        max_c = avg_c + round(random.uniform(2.3,3.0),2)
        temp = random.randint(50,55)
        risk = 1


    fluct = abs(avg_v - min_v)


    # Add small noise (to avoid perfect learning)
    if random.random() < 0.15:   # 15% noise
        risk = 0


    data.append([
        avg_v, min_v, avg_c, max_c, temp, fluct, risk
    ])


df = pd.DataFrame(data, columns=[
    "avg_voltage","min_voltage","avg_current",
    "max_current","temp","fluctuation","risk"
])

df.to_csv("zoneB_data.csv", index=False)

print("Zone B dataset created (70–95% risk target)")


Zone B dataset created (70–95% risk target)


In [57]:
#check for Zone A
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib


# -------------------------------
# CONFIG
# -------------------------------

ZONE = "A"   # Change to "B" for Zone B

DATA_FILE = "zoneA_data.csv" if ZONE=="A" else "zoneB_data.csv"


# -------------------------------
# 1. Load Data
# -------------------------------

df = pd.read_csv(DATA_FILE)

print(f"\n--- Running Analysis for Zone {ZONE} ---")
print("Data Loaded. Total rows:", len(df))


# -------------------------------
# 2. Prepare Data
# -------------------------------

X = df.drop("risk", axis=1)
y = df["risk"]


# -------------------------------
# 3. Train-Test Split
# -------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# -------------------------------
# 4. Train Model
# -------------------------------

model = RandomForestClassifier(n_estimators=100)

model.fit(X_train, y_train)


# -------------------------------
# 5. Check Accuracy
# -------------------------------

accuracy = model.score(X_test, y_test)

print("Model Accuracy:", round(accuracy*100,2), "%")


# -------------------------------
# 6. Save Model
# -------------------------------

joblib.dump(model, f"zone{ZONE}_model.pkl")

print(f"Model Saved as zone{ZONE}_model.pkl")


# -------------------------------
# 7. Predict Using Latest Data
# -------------------------------

latest_data = X.iloc[-1:]   # last row

probs = model.predict_proba(latest_data)[0]

if len(probs) == 1:
    risk_percent = 0.0
else:
    risk_percent = round(probs[1]*100,2)


# -------------------------------
# 8. Decide Status
# -------------------------------

if risk_percent < 15:
    status = "SAFE (Low Risk)"
elif risk_percent < 40:
    status = "SAFE (Monitor)"
elif risk_percent < 70:
    status = "WARNING"
else:
    status = "CRITICAL"


# -------------------------------
# 9. Show Result
# -------------------------------

print("\n--- Latest Area Status ---")
print("Zone:", ZONE)
print("Risk:", risk_percent, "%")
print("Status:", status)


Balanced Zone B dataset created

--- Running Analysis for Zone A ---
Data Loaded. Total rows: 600
Model Accuracy: 88.33 %
Model Saved as zoneA_model.pkl

--- Latest Area Status ---
Zone: A
Risk: 22.0 %
Status: SAFE (Monitor)


In [88]:
#check for Zone B
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib


# -------------------------------
# CONFIG
# -------------------------------

ZONE = "B"   # Change to "B" for Zone B

DATA_FILE = "zoneA_data.csv" if ZONE=="A" else "zoneB_data.csv"


# -------------------------------
# 1. Load Data
# -------------------------------

df = pd.read_csv(DATA_FILE)

print(f"\n--- Running Analysis for Zone {ZONE} ---")
print("Data Loaded. Total rows:", len(df))


# -------------------------------
# 2. Prepare Data
# -------------------------------

X = df.drop("risk", axis=1)
y = df["risk"]


# -------------------------------
# 3. Train-Test Split
# -------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# -------------------------------
# 4. Train Model
# -------------------------------

model = RandomForestClassifier(n_estimators=100)

model.fit(X_train, y_train)


# -------------------------------
# 5. Check Accuracy
# -------------------------------

accuracy = model.score(X_test, y_test)

print("Model Accuracy:", round(accuracy*100,2), "%")


# -------------------------------
# 6. Save Model
# -------------------------------

joblib.dump(model, f"zone{ZONE}_model.pkl")

print(f"Model Saved as zone{ZONE}_model.pkl")


# -------------------------------
# 7. Predict Using Latest Data
# -------------------------------

# Use last 5 records (smoother prediction)
latest_data = X.tail(10).mean().to_frame().T

probs = model.predict_proba(latest_data)[0]

if len(probs) == 1:
    risk_percent = 0.0
else:
    risk_percent = round(probs[1]*100,2)


# -------------------------------
# 8. Decide Status
# -------------------------------

if risk_percent < 50:
    status = "WARNING (Low)"
elif risk_percent < 70:
    status = "WARNING (High)"
else:
    status = "CRITICAL"

# -------------------------------
# 9. Show Result
# -------------------------------

print("\n--- Latest Area Status ---")
print("Zone:", ZONE)
print("Risk:", risk_percent, "%")
print("Status:", status)



--- Running Analysis for Zone B ---
Data Loaded. Total rows: 600
Model Accuracy: 85.83 %
Model Saved as zoneB_model.pkl

--- Latest Area Status ---
Zone: B
Risk: 93.0 %
Status: CRITICAL
